# **Hello**

****This is a beginner friendly intermediate level notebook ;)****

****Please do comment and spread your knowledge. Enlighten me and others.****

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e7/sample_submission.csv
/kaggle/input/playground-series-s5e7/train.csv
/kaggle/input/playground-series-s5e7/test.csv


# ***So here we start.***

In [2]:
#importing useful libraries
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [3]:
#Loading Datasets

X_full = pd.read_csv('../input/playground-series-s5e7/train.csv')
X_test = pd.read_csv('../input/playground-series-s5e7/test.csv')
X_test_og = X_test
"""X_full = X_full.dropna()"""

'X_full = X_full.dropna()'

In [4]:
X_test.isna().sum()

id                             0
Time_spent_Alone             425
Stage_fear                   598
Social_event_attendance      397
Going_outside                466
Drained_after_socializing    432
Friends_circle_size          350
Post_frequency               408
dtype: int64

Some columns have categorical data and for now I don't have much expertize in dealing with categorical data, hence it is better to turn them into numerical ones.

In [5]:
#updating categorical data into binary terms

def turn_to_numeric(Dataframe, column, string):
    new = Dataframe[column] == string
    new += 0
    Dataframe[column] = new
# 1 means Yes, 0 implies No
turn_to_numeric(X_full, 'Stage_fear', 'No')
turn_to_numeric(X_full, 'Drained_after_socializing', 'No')
turn_to_numeric(X_test, 'Stage_fear', 'No')
turn_to_numeric(X_test, 'Drained_after_socializing', 'No')

# 1 means Exrovert and 0 refers to Introvert
turn_to_numeric(X_full, 'Personality', 'Extrovert')


Finally splitting the transformed data.

In [6]:
# Removing not needed columns before splitting the data

y = X_full["Personality"]
X = X_full.drop(["id", "Personality"], axis = 1)


X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2,
                                                      random_state=6)

# Imputation

Imputation for the NAN values.
Since the test data also contains NAN values, it is not a good idea to drop that data.

I will also check in some other version for putting 0 or something like that for NAN values. In that case, I will need to increase the weights of other values.

While there are ways by which we may create models without the need for imputation.
The link to the discussion where it is mentioned is : [How to go without imputing the values](https://www.kaggle.com/competitions/playground-series-s5e7/discussion/593875) suggestions provided by [Sylvester d'Almeida](https://www.kaggle.com/sylvesterdalmeida).



In [7]:
my_imputer = SimpleImputer()
"""This was required in previous versions where NAN values existed"""

# Imputation

imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid))

# Imputation removed column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

# Creating the model

In [8]:
##### model creation
model = RandomForestClassifier(n_estimators = 180, min_samples_split = 15, max_depth = 10,
                               random_state = 3)
model.fit(imputed_X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_split=15, n_estimators=180,
                       random_state=3)

# Model Valiation

Model is created, let's now test it.

In [9]:
test_pred = model.predict(imputed_X_valid)
test_pred

"""
# In case of regression models such thing would have been needed.
# Since it is a classification problem.
for i in range(len(test_pred)):
    test_pred[i] = 1 if test_pred[i]>= 0.28 else 0
"""

'\n# In case of regression models such thing would have been needed.\n# Since it is a classification problem.\nfor i in range(len(test_pred)):\n    test_pred[i] = 1 if test_pred[i]>= 0.28 else 0\n'

**Earlier when I used regression model**

(Our model made really nice predictions, but they were all values ranging from 0 to 1. While we just wanted only two values that are 0 and 1. So we converted those having value 0.5 and above to 1.)


Nice!! But we will require some thing to measure if our model really worked well

In [10]:
accuracy_score(y_valid, test_pred)

0.965991902834008

The accuracy check can also be done manually as below

In [11]:
n = sum((y_valid - test_pred) !=0)
100 - (n / len(y_valid) * 100)

96.59919028340082

# Training on Whole Data

Now as we know how our model need to be, why not train it on whole train data

In [12]:
imputed_X_full = pd.DataFrame(my_imputer.fit_transform(X_full))
imputed_X_full.columns = X_full.columns
imputed_X_full = imputed_X_full.drop(['id', 'Personality'], axis = 1)

model.fit(imputed_X_full, y)

RandomForestClassifier(max_depth=10, min_samples_split=15, n_estimators=180,
                       random_state=3)

Great!!! Now let's make our model work on real test data.

In [13]:
# taking a glance at the test data
X_test.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency
0,18524,3.0,1,7.0,4.0,1,6.0,NaN
1,18525,NaN,0,0.0,0.0,0,5.0,1.0
2,18526,3.0,1,5.0,6.0,1,15.0,9.0
3,18527,3.0,1,4.0,4.0,1,5.0,6.0
4,18528,9.0,0,1.0,2.0,0,1.0,1.0


The test data has some missing values, so let's impute them.

In [14]:
imputed_X_test = pd.DataFrame(my_imputer.fit_transform(X_test))
imputed_X_test.columns = X_test.columns
imputed_X_test = imputed_X_test.drop(['id'], axis = 1)
final_prediction = model.predict(imputed_X_test)

final_prediction

array([1, 0, 1, ..., 1, 1, 0])

# Final Predictions

Final Predictions are made, now let's convert it into the asked format.

In [15]:
list = []
for i in range(len(final_prediction)):
    if final_prediction[i]>=0.5:
        list.append("Extrovert")
    else :
        list.append("Introvert")

output = pd.DataFrame({'id': X_test_og.id,
                       'Personality': list})
output.to_csv('submission.csv', index=False)

Awesome!! That's it, just one more line of code. (To see how our output looks like)

In [16]:
len(X_test)
output.groupby("Personality").count()

,id
Personality,
Extrovert,4622
Introvert,1553


Thank you for coming down till here.

If you reached here, do consider giving an upvote.